# Multimodal RAG Demo — Niwas Housing Finance FY2025

**Assignment:** AI Engineering Take-Home — Multimodal RAG on the Niwas FY2025 Annual Report  
**Document:** NHFPL Annual Report FY2025 (133 pages)  
**Pipeline:** PyMuPDF parsing → BGE embeddings → ChromaDB → Groq LLaMA-3.3-70b  

This notebook answers all 6 required test questions and shows:
- **(a)** Retrieved chunks (rank, page, content type, score, preview)
- **(b)** Final answer with source citations


## Setup

In [2]:
import sys
import os
from pathlib import Path

# ── Always resolve to project root (parent of notebooks/) ────────────
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)   # so data/, outputs/ resolve correctly

print(f'Project root : {PROJECT_ROOT}')
print(f'Python       : {sys.version}')

Project root : /Users/sakshizanjad/Desktop/Projects/multimodal_rag
Python       : 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.4.2)]


In [3]:
import config as cfg
from src.rag_engine import RAGEngine, RAGAnswer

print(f'LLM Provider : {cfg.LLM_PROVIDER}')
print(f'LLM Model    : {cfg.GROQ_LLM_MODEL if cfg.LLM_PROVIDER == "groq" else cfg.OLLAMA_LLM_MODEL}')
print(f'Embedding    : {cfg.EMBEDDING_MODEL}')
print(f'Top-K        : {cfg.TOP_K}')
print(f'Vector DB    : {cfg.VECTORDB_DIR}')

LLM Provider : groq
LLM Model    : llama-3.3-70b-versatile
Embedding    : BAAI/bge-base-en-v1.5
Top-K        : 5
Vector DB    : /Users/sakshizanjad/Desktop/Projects/multimodal_rag/data/vectordb


## Initialise RAG Engine

In [4]:
engine = RAGEngine()
print('RAG Engine ready.')

/Users/sakshizanjad/Desktop/Projects/multimodal_rag/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


00:01:24 | INFO     | src.embedder | Loading embedding model: BAAI/bge-base-en-v1.5
00:01:29 | INFO     | src.embedder | Embedding model loaded.
00:01:29 | INFO     | src.retriever | Retriever ready — collection has 411 docs
00:01:29 | INFO     | src.rag_engine | Groq LLM: llama-3.3-70b-versatile
RAG Engine ready.


## Helper — Display Function

In [5]:
from IPython.display import display, Markdown
import time

INTER_QUESTION_DELAY = cfg.LLM_INTER_QUESTION_DELAY

def ask_and_display(question_id, question_type, question, source_hint, delay=False):
    if delay and cfg.LLM_PROVIDER == 'groq':
        print(f'  Waiting {INTER_QUESTION_DELAY}s (Groq TPM limit) ...')
        time.sleep(INTER_QUESTION_DELAY)

    display(Markdown(f'---\n## Q{question_id} [{question_type}]'))
    display(Markdown(f'**Question:** {question}  \n**Source hint:** `{source_hint}`'))

    answer = engine.ask(question)

    display(Markdown(f'### Retrieved Chunks ({len(answer.retrieved_chunks)} total)'))
    type_icon = {'text': 'TEXT', 'table': 'TABLE', 'image': 'IMAGE'}
    rows = ['| Rank | Type | Page | Section | Score | Preview |',
            '|------|------|------|---------|-------|---------|']
    for c in answer.retrieved_chunks:
        prev = c['text'][:100].replace('\n', ' ').replace('|', '/')
        rows.append(f"| {c['rank']} | {type_icon.get(c['content_type'], c['content_type'])} | {c['page_num']} | {c['section']} | {c['score']:.4f} | {prev}... |")
    display(Markdown('\n'.join(rows)))

    display(Markdown('### Answer'))
    display(Markdown(answer.answer))
    display(Markdown('### Citations'))
    for cite in answer.citations:
        display(Markdown(f'- {cite}'))
    display(Markdown(f'*{answer.elapsed_seconds:.2f}s | {answer.provider}/{answer.model}*'))
    return answer

print('Helper ready.')

Helper ready.


---
# Required Test Questions

## Q1 — TEXT: Key Risks (MD&A)

In [6]:
ans1 = ask_and_display(1, 'TEXT',
    "What are the key risks to Niwas's business highlighted in the MD&A section?",
    'MD&A, pages 12-18', delay=False)

---
## Q1 [TEXT]

**Question:** What are the key risks to Niwas's business highlighted in the MD&A section?  
**Source hint:** `MD&A, pages 12-18`

00:01:40 | INFO     | src.rag_engine | Question: What are the key risks to Niwas's business highlighted in the MD&A section?
00:01:41 | INFO     | src.rag_engine | Retrieved 5 chunks from pages: [41, 17, 29, 11, 103]
00:01:42 | INFO     | src.rag_engine | Answer generated in 2.10s


### Retrieved Chunks (5 total)

| Rank | Type | Page | Section | Score | Preview |
|------|------|------|---------|-------|---------|
| 1 | IMAGE | 41 | Board's Report & Governance | 0.3866 | [Visual content on page 41] The image presents a page from an annual report, specifically focusing o... |
| 2 | TEXT | 17 | Management Discussion & Analysis | 0.3870 | ional risk arising from business operations.  The Committee evaluates the appropriateness of  major ... |
| 3 | IMAGE | 29 | Board's Report & Governance | 0.3958 | [Visual content on page 29] The image presents a page from an annual report, specifically focusing o... |
| 4 | IMAGE | 11 | Corporate Overview | 0.3986 | [Visual content on page 11] The image presents a page from an annual report, specifically titled "Co... |
| 5 | IMAGE | 103 | Notes to Accounts | 0.3999 | [Visual content on page 103] The image presents a page from an annual report, specifically "Notes to... |

...

- Source: Management Discussion & Analysis, page 17

- Source: Figure/Chart from, Board's Report & Governance, page 29

- Source: Figure/Chart from, Corporate Overview, page 11

- Source: Figure/Chart from, Notes to Accounts, page 103

*2.10s | groq/llama-3.3-70b-versatile*

## Q2 — TEXT: Strategy (Board's Report)

In [7]:
ans2 = ask_and_display(2, 'TEXT',
    "What is Niwas's stated strategy for scaling its loan book as described in the Board's Report?",
    "Board's Report, pages 19-24", delay=True)

  Waiting 30s (Groq TPM limit) ...


---
## Q2 [TEXT]

**Question:** What is Niwas's stated strategy for scaling its loan book as described in the Board's Report?  
**Source hint:** `Board's Report, pages 19-24`

00:02:23 | INFO     | src.rag_engine | Question: What is Niwas's stated strategy for scaling its loan book as described in the Board's Report?
00:02:23 | INFO     | src.rag_engine | Retrieved 5 chunks from pages: [5, 10, 11, 29, 1]
00:02:24 | INFO     | src.rag_engine | Answer generated in 1.25s


### Retrieved Chunks (5 total)

...

- Source: Figure/Chart from, Corporate Overview, page 10

- Source: Figure/Chart from, Corporate Overview, page 11

- Source: Figure/Chart from, Board's Report & Governance, page 29

- Source: Figure/Chart from, Corporate Overview, page 1

*1.25s | groq/llama-3.3-70b-versatile*

## Q3 — TABLE: Total Loan Book (Balance Sheet)

In [8]:
ans3 = ask_and_display(3, 'TABLE',
    'What is the total loan book (Loans line) as at March 31, 2025 vs March 31, 2024?',
    'Balance Sheet, page 61', delay=True)

  Waiting 30s (Groq TPM limit) ...


---
## Q3 [TABLE]

**Question:** What is the total loan book (Loans line) as at March 31, 2025 vs March 31, 2024?  
**Source hint:** `Balance Sheet, page 61`

00:03:03 | INFO     | src.rag_engine | Question: What is the total loan book (Loans line) as at March 31, 2025 vs March 31, 2024?
00:03:03 | INFO     | src.rag_engine | Retrieved 5 chunks from pages: [106, 124, 87, 125, 86]
00:03:04 | INFO     | src.rag_engine | Answer generated in 1.36s


### Retrieved Chunks (5 total)

...

- Source: Notes to Accounts, page 124

- Source: Notes to Accounts, page 87

- Source: Notes to Accounts, page 125

- Source: Notes to Accounts, page 86

*1.36s | groq/llama-3.3-70b-versatile*

## Q4 — TABLE: Interest Income (P&L)

In [9]:
ans4 = ask_and_display(4, 'TABLE',
    "What was Niwas's total interest income for FY2025 and how did it compare to FY2024?",
    'P&L, page 62', delay=True)

  Waiting 30s (Groq TPM limit) ...


---
## Q4 [TABLE]

**Question:** What was Niwas's total interest income for FY2025 and how did it compare to FY2024?  
**Source hint:** `P&L, page 62`

00:03:45 | INFO     | src.rag_engine | Question: What was Niwas's total interest income for FY2025 and how did it compare to FY2024?
00:03:45 | INFO     | src.rag_engine | Retrieved 5 chunks from pages: [63, 3, 91, 87, 61]
00:03:46 | INFO     | src.rag_engine | Answer generated in 1.77s


### Retrieved Chunks (5 total)

...

- Source: Figure/Chart from, Corporate Overview, page 3

- Source: Figure/Chart from, Notes to Accounts, page 91

- Source: Figure/Chart from, Notes to Accounts, page 87

- Source: Figure/Chart from, Financial Statements, page 61

*1.77s | groq/llama-3.3-70b-versatile*

## Q5 — IMAGE: Geographic Loan Portfolio

In [10]:
ans5 = ask_and_display(5, 'IMAGE',
    'Describe the trend or distribution shown in the geographic / state-wise loan portfolio figure.',
    'pages 15-16', delay=True)

  Waiting 30s (Groq TPM limit) ...


---
## Q5 [IMAGE]

**Question:** Describe the trend or distribution shown in the geographic / state-wise loan portfolio figure.  
**Source hint:** `pages 15-16`

00:04:35 | INFO     | src.rag_engine | Question: Describe the trend or distribution shown in the geographic / state-wise loan portfolio figure.
00:04:35 | INFO     | src.rag_engine | Retrieved 5 chunks from pages: [77, 15, 14, 100, 78]
00:04:36 | INFO     | src.rag_engine | Answer generated in 1.20s


### Retrieved Chunks (5 total)

...

- Source: Figure/Chart from, Management Discussion & Analysis, page 15

- Source: Figure/Chart from, Management Discussion & Analysis, page 14

- Source: Figure/Chart from, Notes to Accounts, page 100

- Source: Figure/Chart from, Notes to Accounts, page 78

*1.20s | groq/llama-3.3-70b-versatile*

## Q6 — IMAGE: AUM / Disbursement Chart

In [11]:
ans6 = ask_and_display(6, 'IMAGE',
    'What does the AUM or disbursement chart in the Corporate Overview section show about growth over the years?',
    'Corporate Overview, pages 3-9', delay=True)

  Waiting 30s (Groq TPM limit) ...


---
## Q6 [IMAGE]

**Question:** What does the AUM or disbursement chart in the Corporate Overview section show about growth over the years?  
**Source hint:** `Corporate Overview, pages 3-9`

00:05:12 | INFO     | src.rag_engine | Question: What does the AUM or disbursement chart in the Corporate Overview section show about growth over the years?
00:05:12 | INFO     | src.rag_engine | Retrieved 5 chunks from pages: [7, 24, 78, 79, 99]
00:05:13 | INFO     | src.rag_engine | Answer generated in 1.41s


### Retrieved Chunks (5 total)

...

- Source: Figure/Chart from, Board's Report & Governance, page 24

- Source: Figure/Chart from, Notes to Accounts, page 78

- Source: Figure/Chart from, Notes to Accounts, page 79

- Source: Figure/Chart from, Notes to Accounts, page 99

*1.41s | groq/llama-3.3-70b-versatile*

---
## Summary

In [12]:
import json
from datetime import datetime

questions = [
    {'id': 1, 'type': 'TEXT',  'source_hint': 'MD&A, pages 12-18',            'question': "What are the key risks to Niwas's business highlighted in the MD&A section?"},
    {'id': 2, 'type': 'TEXT',  'source_hint': "Board's Report, pages 19-24",  'question': "What is Niwas's stated strategy for scaling its loan book as described in the Board's Report?"},
    {'id': 3, 'type': 'TABLE', 'source_hint': 'Balance Sheet, page 61',       'question': 'What is the total loan book (Loans line) as at March 31, 2025 vs March 31, 2024?'},
    {'id': 4, 'type': 'TABLE', 'source_hint': 'P&L, page 62',                 'question': "What was Niwas's total interest income for FY2025 and how did it compare to FY2024?"},
    {'id': 5, 'type': 'IMAGE', 'source_hint': 'pages 15-16',                  'question': 'Describe the trend or distribution shown in the geographic / state-wise loan portfolio figure.'},
    {'id': 6, 'type': 'IMAGE', 'source_hint': 'Corporate Overview, pages 3-9','question': 'What does the AUM or disbursement chart in the Corporate Overview section show about growth over the years?'},
]
answers = [ans1, ans2, ans3, ans4, ans5, ans6]

display(Markdown('### Results Summary'))
rows = ['| Q | Type | Pages Retrieved | Time |', '|---|------|----------------|------|']
for q, a in zip(questions, answers):
    pages = sorted(set(c['page_num'] for c in a.retrieved_chunks))
    rows.append(f"| Q{q['id']} | {q['type']} | {pages} | {a.elapsed_seconds:.1f}s |")
display(Markdown('\n'.join(rows)))

results = [{'question_id': q['id'], 'type': q['type'], 'source_hint': q['source_hint'], **a.to_dict()}
           for q, a in zip(questions, answers)]
out_path = Path('outputs') / f"notebook_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(results, ensure_ascii=False, indent=2))
print(f'Results saved to {out_path}')

### Results Summary

| Q | Type | Pages Retrieved | Time |
|---|------|----------------|------|
| Q1 | TEXT | [11, 17, 29, 41, 103] | 2.1s |
| Q2 | TEXT | [1, 5, 10, 11, 29] | 1.2s |
| Q3 | TABLE | [86, 87, 106, 124, 125] | 1.4s |
| Q4 | TABLE | [3, 61, 63, 87, 91] | 1.8s |
| Q5 | IMAGE | [14, 15, 77, 78, 100] | 1.2s |
| Q6 | IMAGE | [7, 24, 78, 79, 99] | 1.4s |

Results saved to outputs/notebook_results_20260622_000552.json


---
## Pipeline Architecture

```
PDF (133 pages)
      │
      ├─ Phase 1: PyMuPDF Parser       → parsed.json
      ├─ Phase 2: Image Extractor      → page renders + embedded images
      ├─ Phase 3: Vision Page Selector → vision_pages.json (100/133 pages selected)
      ├─ Phase 4: Vision Analyzer      → vision_descriptions.json (Groq Llama-4-Scout)
      ├─ Phase 5: Multimodal Chunker   → chunks.json (text + table + image chunks)
      ├─ Phase 6: BGE Embedder         → BAAI/bge-base-en-v1.5 embeddings
      ├─ Phase 7: ChromaDB Indexer     → vectordb/
      ├─ Phase 8: Semantic Retriever   → top-5 chunks per query
      └─ Phase 9: RAG Engine           → Groq LLaMA-3.3-70b → grounded answer
```

**Key decisions:**
- PyMuPDF for fast text + block extraction
- Groq Llama-4-Scout for vision understanding of charts/tables
- BAAI/bge-base-en-v1.5 for asymmetric retrieval
- ChromaDB local vector store (zero config)
- Boilerplate page headers stripped before chunking for cleaner BM25 signal
